# Test the SDXL + IP-Adapter worker on Colab
Runtime → Change runtime type → **T4 GPU** (free tier works). Runs `handler.py` unchanged, so whatever passes here should behave the same on RunPod.

Upload `handler.py` and `requirements.txt` with the Files panel (left sidebar) before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -r requirements.txt

## 1. Load the model (this is the cold-start cost)

In [ ]:
import os
os.environ['LOW_VRAM'] = '1'  # T4 has ~15GB; not needed on 24GB+ RunPod GPUs
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import time, base64, io
t = time.perf_counter()
import handler  # loads SDXL + IP-Adapter at import, same as on RunPod
print(f'cold load: {time.perf_counter() - t:.1f}s')

## 2. Text-only request

In [ ]:
from PIL import Image
from IPython.display import display

def run(inp):
    out = handler.handler({'input': inp})
    if 'error' in out:
        raise RuntimeError(out['error'])
    print('seed', out['seed'], {k: round(v) for k, v in out['timings_ms'].items()})
    img = Image.open(io.BytesIO(base64.b64decode(out['image_base64'])))
    display(img.resize((512, 512)))
    return out

_ = run({'prompt': 'a corgi wearing a tiny astronaut helmet on the moon, cinematic, funny', 'seed': 42})

## 3. With a pet photo (IP-Adapter)
Upload a pet photo as `pet.jpg`. Try a few `ip_adapter_scale` values: higher keeps more identity, lower follows the prompt more.

In [ ]:
pet_b64 = base64.b64encode(open('pet.jpg', 'rb').read()).decode()
for scale in [0.4, 0.6, 0.8]:
    print('ip_adapter_scale', scale)
    _ = run({'prompt': 'this dog as a pirate captain on a ship, funny, cinematic', 'pet_image': pet_b64,
             'ip_adapter_scale': scale, 'seed': 42})

## 4. Quick warm-latency benchmark

In [ ]:
import statistics
lat = [handler.handler({'input': {'prompt': 'a cat in a business suit', 'steps': 30, 'seed': i}})['timings_ms']['inference'] for i in range(5)]
print(f'inference p50 {statistics.median(lat):.0f} ms, max {max(lat):.0f} ms over {len(lat)} runs')

## 5. (Optional) Expose as a RunPod-style API for the proxy server

⚠️ **Sections 1–4 and section 5 can't run in the same session on a T4.** Sections 1–4 load SDXL into this notebook (~14 GB of GPU memory), and this cell loads it again in a separate process. To use the tunnel: **Runtime → Restart session**, then run *only* this cell. `nvidia-smi` should show ~0 MiB used before you start it.

The runpod SDK serves the handler with the same `/run` + `/status` routes as RunPod, and a cloudflared tunnel gives it a public URL. Put that URL in `RUNPOD_BASE_URL` in the server's `.env`. Wait for `Uvicorn running on http://0.0.0.0:8000` (2–4 min) before sending requests. This cell blocks the notebook while it runs.

In [ ]:
import os, subprocess, time
used = int(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits']).decode().strip())
assert used < 1000, f'{used} MiB of GPU memory already in use: restart the session and run only this cell'
os.environ['LOW_VRAM'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stderr=open('tunnel.log', 'w'))
time.sleep(8)
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | head -1
!python handler.py --rp_serve_api --rp_api_host 0.0.0.0 --rp_api_port 8000